# Supply Chain Logistics Analysis
### Lead-Time Bottlenecks in a Multi-Echelon Logistics Network

**Data Science Bootcamp #15 capstone** | Team: [add names] | Code: [GitHub link] | Dashboard: [Tableau link]

Dataset: [Supply Chain Logistics Problem](https://www.kaggle.com/datasets/anisseezzebdi/supply-chain-logistics-problem) (OrderList table). Port, plant and carrier names are anonymised codes.

## TL;DR
*Values below come from the project run. Confirm them against this notebook's output after **Run All**.*
- 4,899 shipment records analysed (from 9,215 raw OrderList rows).
- About 97% of shipments use one origin port (PORT04): a concentration risk.
- Carrier and plant significantly affect transit processing time (ANOVA, Tukey HSD).
- Random Forest: RMSE 1.32 days, MAE 1.08, R² 0.29, a baseline with limited predictive power.

## Contents
1. Setup · 2. Load and inspect · 3. Clean · 4. Exploratory analysis · 5. Statistical testing · 6. Predictive modelling · 7. Findings and recommendations · 8. Limitations and next steps

## How to run
1. On Kaggle, add the dataset with **Add Input** (search *Supply Chain Logistics Problem*). Internet is not needed.
2. Run all cells top to bottom. The XGBoost grid search takes a few minutes; set `FAST_MODE = True` in section 1 for a quick test.
3. Outside Kaggle, set the `ORDERS_PATH` environment variable to the Excel file, or install `kagglehub` and let the notebook download it.

## 1. Setup

In [ ]:
import glob
import os
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import scipy.stats as stats
import seaborn as sns
import sklearn
import statsmodels.api as sm
import xgboost
from IPython.display import display
from plotly.subplots import make_subplots
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GridSearchCV, train_test_split
from statsmodels.stats.multicomp import pairwise_tukeyhsd

warnings.filterwarnings("ignore", category=FutureWarning)

print(
    f"pandas {pd.__version__} | numpy {np.__version__} | "
    f"scikit-learn {sklearn.__version__} | xgboost {xgboost.__version__}"
)

In [ ]:
# --- Configuration: change values here, not inside the analysis cells ---
SEED = 42
ALPHA = 0.05
MAX_RECORDS = 5000  # project size guideline: 1,000 to 5,000 records
EXPECTED_RECORDS = 4899  # record count reported in the project report
TEST_SIZE = 0.2
FAST_MODE = False  # True shrinks the XGBoost grid for a quick test run

DATASET_SLUG = "anisseezzebdi/supply-chain-logistics-problem"
EXCEL_NAME = "Supply chain logistics problem.xlsx"
SHEET_NAME = "OrderList"

TARGET = "Total_Lead_Time"
FEATURES = ["Weight", "Carrier", "Origin Port", "Plant Code", "Service Level", "Unit quantity"]
REQUIRED_COLUMNS = set(FEATURES) | {"Order Date", "TPT", "Ship ahead day count", "Product ID"}

COLORS = {
    "primary": "#005b96",
    "accent": "#ff9900",
    "muted": "#6497b1",
    "danger": "#d9534f",
    "grey": "#cccccc",
}
sns.set_theme(style="whitegrid", rc={"axes.edgecolor": COLORS["grey"], "axes.linewidth": 1})

In [ ]:
def find_orders_file() -> str:
    """Return the path of the Excel file (override, Kaggle input folder, or kagglehub)."""
    override = os.environ.get("ORDERS_PATH")
    if override:
        return override
    hits = glob.glob("/kaggle/input/**/*.xlsx", recursive=True)
    if hits:
        return hits[0]
    try:
        import kagglehub  # only needed outside Kaggle, for example in Colab

        return os.path.join(kagglehub.dataset_download(DATASET_SLUG), EXCEL_NAME)
    except Exception as exc:  # any download failure gets one clear message
        raise FileNotFoundError(
            "Orders file not found. On Kaggle use Add Input > Supply Chain Logistics Problem."
        ) from exc


def evaluate_model(name: str, y_true, y_pred) -> dict:
    """Return MAE, RMSE and R2 for one model on the shared test set."""
    return {
        "Model": name,
        "MAE (Days)": mean_absolute_error(y_true, y_pred),
        "RMSE (Days)": float(np.sqrt(mean_squared_error(y_true, y_pred))),
        "R2": r2_score(y_true, y_pred),
    }


def add_note(ax, text: str, x: float = 0.03, y: float = 0.95, ha: str = "left") -> None:
    """Place a rounded annotation box in axes coordinates."""
    ax.text(
        x, y, text, transform=ax.transAxes, ha=ha, va="top", fontsize=10, style="italic",
        bbox={"facecolor": "#fef7e5", "alpha": 0.9, "edgecolor": "#ffcc66",
              "boxstyle": "round,pad=0.5"},
    )


def label_bars(ax, fmt: str = "%.0f") -> None:
    """Write the value above every bar on an axes."""
    for container in ax.containers:
        ax.bar_label(container, fmt=fmt, padding=3, fontsize=9)

## 2. Load and inspect the data

In [ ]:
orders_raw = pd.read_excel(find_orders_file(), sheet_name=SHEET_NAME)
print(f"Records: {len(orders_raw):,} | Features: {orders_raw.shape[1]}\n")
orders_raw.info()

nulls = orders_raw.isna().sum()
print("\nColumns with missing values:")
display(nulls[nulls > 0].to_frame("Missing Count"))
display(orders_raw.describe().T)
display(orders_raw.head(3))
print("Distinct Order Date values:", orders_raw["Order Date"].nunique())

**Data note:** the source `Order Date` column holds a single distinct value (see the last line above). The analysis is therefore **cross-sectional**: there is no trend or seasonality analysis, and "most recent records" has no real meaning in the sampling step below.

## 3. Clean the data

In [ ]:
def clean_orders(raw: pd.DataFrame, max_records: int = MAX_RECORDS) -> pd.DataFrame:
    """Clean the OrderList table. Steps and their order match the project report."""
    missing = REQUIRED_COLUMNS - set(raw.columns)
    if missing:
        raise KeyError(f"Missing required columns: {sorted(missing)}")

    df = raw.copy()
    audit = [("Raw records", len(df))]

    df["Weight"] = df["Weight"].fillna(df["Weight"].median())  # median resists outliers
    df["Order Date"] = pd.to_datetime(df["Order Date"])
    # Keep this default sort. Every Order Date is identical, so this call decides which rows
    # survive the cap below; kind="stable" would change the sample and all reported numbers.
    df = df.sort_values(by="Order Date")

    df = df.drop_duplicates()
    audit.append(("After de-duplication", len(df)))

    df[TARGET] = df["TPT"] + df["Ship ahead day count"]
    if len(df) > max_records:
        df = df.tail(max_records)
        audit.append((f"Capped at {max_records:,} (project size guideline)", len(df)))

    low, high = df["Weight"].quantile(0.01), df["Weight"].quantile(0.99)
    df = df[(df["Weight"] > low) & (df["Weight"] < high)]
    audit.append(("After Weight trimming (1st-99th percentile)", len(df)))

    # Delivery_Delay equals the "Ship ahead day count" component of lead time, so its
    # correlation with Total_Lead_Time is built in by definition.
    df["Delivery_Delay"] = df[TARGET] - df["TPT"]

    print(pd.DataFrame(audit, columns=["Step", "Records"]).to_string(index=False))
    return df


cleaned = clean_orders(orders_raw)
if len(cleaned) != EXPECTED_RECORDS:
    print(
        f"\nNOTE: {len(cleaned):,} records, but the report states {EXPECTED_RECORDS:,}. All Order "
        "Dates are identical, so the sort that picks the 5,000 rows can differ between library "
        "versions. Expect small shifts in the downstream numbers."
    )
display(cleaned.head())

## 4. Exploratory analysis and KPIs
KPIs covered: delivery lead time, transit processing time (TPT), delay, shipment volume by node, and weight.

In [ ]:
def plot_kpi_overview(df: pd.DataFrame) -> None:
    """Draw five KPI panels: port volume vs lead time, lead-time distribution, carrier TPT
    spread, plant delay, and top products by weight."""
    port = df.groupby("Origin Port").agg(
        shipments=(TARGET, "size"), avg_lead=(TARGET, "mean")
    ).reset_index()
    fig = plt.figure(figsize=(16, 12), layout="constrained")
    grid = fig.add_gridspec(3, 2, height_ratios=[1.2, 1, 1])

    # A: volume (bars) and average lead time (line) by origin port
    ax = fig.add_subplot(grid[0, :])
    pos = np.arange(len(port))
    ax.bar(pos, port["shipments"], color=COLORS["primary"], alpha=0.85)
    label_bars(ax)
    ax.set_xticks(pos, port["Origin Port"])
    ax.set_ylabel("Shipments", color=COLORS["primary"])
    ax_lead = ax.twinx()
    ax_lead.plot(pos, port["avg_lead"], color=COLORS["accent"], marker="o", lw=2.5)
    ax_lead.grid(False)
    ax_lead.set_ylabel("Avg lead time (days)", color=COLORS["accent"])
    for x_pos, value in zip(pos, port["avg_lead"]):
        ax_lead.annotate(f"{value:.2f}d", (x_pos, value), textcoords="offset points",
                         xytext=(0, 8), ha="center", color=COLORS["accent"], fontweight="bold")
    top = port.loc[port["shipments"].idxmax()]
    ax.set_title(f"Hub activity: {top['Origin Port']} handles {top['shipments'] / len(df):.0%} "
                 "of shipments", fontsize=14, fontweight="bold")

    # B: lead-time distribution
    ax = fig.add_subplot(grid[1, 0])
    sns.histplot(df[TARGET], bins=25, kde=True, color=COLORS["muted"], edgecolor="white", ax=ax)
    mean_lead = df[TARGET].mean()
    ax.axvline(mean_lead, color=COLORS["danger"], linestyle="--", linewidth=2)
    ax.set_title(f"Lead time distribution (mean {mean_lead:.2f} days)", fontweight="bold")
    ax.set_xlabel("Total lead time (days)")

    # C: carrier TPT spread, widest spread highlighted
    ax = fig.add_subplot(grid[1, 1])
    names = list(df.groupby("Carrier")["TPT"].std().sort_values(ascending=False).index)
    box = ax.boxplot([df.loc[df["Carrier"] == c, "TPT"] for c in names], patch_artist=True)
    for patch, name in zip(box["boxes"], names):
        patch.set_facecolor(COLORS["accent"] if name == names[0] else COLORS["muted"])
    ax.set_xticklabels(names)
    ax.set_title(f"Carrier TPT spread (widest: {names[0]})", fontweight="bold")
    ax.set_ylabel("TPT (days)")

    # D: average delay by plant, highest highlighted
    ax = fig.add_subplot(grid[2, 0])
    plant_delay = df.groupby("Plant Code")["Delivery_Delay"].mean().sort_values(ascending=False)
    bar_colors = [COLORS["danger"] if i == 0 else COLORS["muted"] for i in range(len(plant_delay))]
    ax.bar(plant_delay.index, plant_delay.values, color=bar_colors)
    label_bars(ax, "%.2f")
    ax.set_title(f"Average delay by plant (highest: {plant_delay.index[0]})", fontweight="bold")
    ax.set_ylabel("Delivery delay (days)")

    # E: top products by total shipped weight
    ax = fig.add_subplot(grid[2, 1])
    top_products = df.groupby("Product ID")["Weight"].sum().nlargest(8)
    ax.bar(top_products.index.astype(str), top_products.values, color=COLORS["primary"], alpha=0.85)
    label_bars(ax)
    ax.tick_params(axis="x", rotation=30)
    ax.set_title("Top 8 products by total shipped weight", fontweight="bold")
    ax.set_ylabel("Total weight")
    plt.show()


plot_kpi_overview(cleaned)

In [ ]:
def plot_eda_panels(df: pd.DataFrame) -> pd.DataFrame:
    """Draw correlation, carrier density, weight trend and port density; return correlations."""
    corr = df[[TARGET, "TPT", "Delivery_Delay", "Weight"]].corr()
    print("Correlation matrix (exact values):")
    print(corr.round(4))

    fig = plt.figure(figsize=(16, 14), layout="constrained")
    grid = fig.add_gridspec(2, 2)

    ax = fig.add_subplot(grid[0, 0])
    mask = np.triu(np.ones(corr.shape, dtype=bool))
    sns.heatmap(corr, mask=mask, cmap=sns.diverging_palette(230, 20, as_cmap=True), vmin=-1,
                vmax=1, center=0, square=True, linewidths=0.5, annot=True, fmt=".2f", ax=ax)
    ax.set_title("Linear relationships (correlation heatmap)", fontsize=14, fontweight="bold")
    add_note(ax, f"TPT vs lead time: r = {corr.loc['TPT', TARGET]:.2f}.\n"
                 "Delivery_Delay is part of lead time by definition.", x=0.95, ha="right")

    ax = fig.add_subplot(grid[0, 1])
    sns.violinplot(data=df, x="Carrier", y=TARGET, color=COLORS["primary"], inner="quartile",
                   linewidth=1.5, ax=ax)
    sns.stripplot(data=df, x="Carrier", y=TARGET, color="black", alpha=0.1, size=2, ax=ax)
    ax.set_title("Carrier performance profile (density and spread)", fontsize=14,
                 fontweight="bold")
    ax.set_ylabel("Total lead time (days)")

    ax = fig.add_subplot(grid[1, 0])
    sns.regplot(data=df, x="Weight", y=TARGET, scatter_kws={"alpha": 0.3, "color": COLORS["muted"]},
                line_kws={"color": COLORS["danger"], "linewidth": 3}, ax=ax)
    ax.set_title("Trend: weight vs lead time", fontsize=14, fontweight="bold")
    add_note(ax, f"Weight vs lead time: r = {corr.loc['Weight', TARGET]:.2f} (weak).")

    ax = fig.add_subplot(grid[1, 1])
    counts = df["Origin Port"].value_counts()
    ports = counts[counts >= 30].index  # density needs enough points per group
    sns.kdeplot(data=df[df["Origin Port"].isin(ports)], x=TARGET, hue="Origin Port", fill=True,
                common_norm=False, alpha=0.5, palette=list(COLORS.values())[:len(ports)], ax=ax)
    ax.set_title("Lead time by origin port (ports with 30+ shipments)", fontsize=14,
                 fontweight="bold")
    plt.suptitle("Exploratory data analysis", fontsize=18, fontweight="bold")
    plt.show()
    return corr


corr_matrix = plot_eda_panels(cleaned)

In [ ]:
def build_dashboard(df: pd.DataFrame) -> go.Figure:
    """Build the interactive four-chart logistics dashboard."""
    port = df.groupby("Origin Port").agg(
        shipments=(TARGET, "size"), avg_lead=(TARGET, "mean")
    ).reset_index().sort_values("shipments", ascending=False)
    plant_tpt = df.groupby("Plant Code")["TPT"].mean().sort_values(ascending=False)
    carriers = df["Carrier"].value_counts().nlargest(10)
    products = df["Product ID"].value_counts().nlargest(10)

    fig = make_subplots(
        rows=2, cols=2,
        specs=[[{"secondary_y": True}, {"type": "bar"}], [{"type": "bar"}, {"type": "bar"}]],
        subplot_titles=("Port impact: volume vs lead time", "Bottlenecks: average TPT by plant",
                        "Vendor reliance: top carriers", "Throughput: top 10 products"),
        vertical_spacing=0.18, horizontal_spacing=0.12,
    )
    fig.add_trace(go.Bar(x=port["Origin Port"], y=port["shipments"], name="Volume",
                         marker_color=COLORS["primary"], opacity=0.85),
                  row=1, col=1, secondary_y=False)
    fig.add_trace(go.Scatter(x=port["Origin Port"], y=port["avg_lead"], name="Avg lead time",
                             mode="lines+markers", marker_color=COLORS["accent"],
                             line={"width": 3}), row=1, col=1, secondary_y=True)
    plant_colors = [COLORS["danger"] if i == 0 else COLORS["muted"] for i in range(len(plant_tpt))]
    fig.add_trace(go.Bar(x=plant_tpt.index.astype(str), y=plant_tpt.values, name="Avg TPT",
                         marker_color=plant_colors), row=1, col=2)
    fig.add_trace(go.Bar(y=carriers.index.astype(str), x=carriers.values, orientation="h",
                         name="Carriers", marker_color=COLORS["primary"], opacity=0.85),
                  row=2, col=1)
    fig.add_trace(go.Bar(x=products.index.astype(str), y=products.values, name="Products",
                         marker_color=COLORS["muted"]), row=2, col=2)

    fig.update_yaxes(autorange="reversed", row=2, col=1)
    fig.update_xaxes(type="category", row=1, col=2, title_text="Plant code")
    fig.update_xaxes(type="category", row=2, col=2, title_text="Product ID")
    fig.update_xaxes(title_text="Origin port", row=1, col=1)
    fig.update_xaxes(title_text="Shipments", row=2, col=1)
    fig.update_yaxes(title_text="Shipments", row=1, col=1, secondary_y=False)
    fig.update_yaxes(title_text="Days", row=1, col=1, secondary_y=True)
    fig.update_layout(title_text="<b>Logistics performance dashboard</b>", height=850,
                      showlegend=False, template="plotly_white",
                      margin={"t": 100, "b": 50, "l": 60, "r": 60})
    return fig


build_dashboard(cleaned).show()

## 5. Statistical testing
Four tests at a significance level of 0.05. Each result is printed and collected into one summary table.

| Test | Question |
|---|---|
| One-way ANOVA | Does carrier change mean transit processing time (TPT)? |
| Welch t-test (one-sided) | Is the plant with the highest average delay worse than the rest of the network? |
| Pearson correlation | Is shipment weight linearly related to total lead time? |
| One-way ANOVA | Does origin port change mean total lead time? (PORT05 has one record, so read this with care.) |

In [ ]:
def verdict(p_value: float) -> str:
    """Translate a p-value into a decision at the configured significance level."""
    return "Reject H0 (significant)" if p_value < ALPHA else "Fail to reject H0"


def show_result(title: str, label: str, statistic: float, p_value: float) -> None:
    """Print one test result in a consistent layout."""
    print(f"\n--- {title} ---")
    print(f"{label}: {statistic:.4f} | p-value: {p_value:.4e} | {verdict(p_value)}")


def run_hypothesis_tests(df: pd.DataFrame) -> dict:
    """Run the four project tests, print each result, and return the statistics."""
    print("=" * 70)
    print(f"{'FORMAL HYPOTHESIS TESTING RESULTS':^70}")
    print("=" * 70)

    carrier_groups = [g["TPT"].dropna() for _, g in df.groupby("Carrier")]
    f_carrier, p_carrier = stats.f_oneway(*carrier_groups)
    show_result("TEST 1: ONE-WAY ANOVA (Carrier vs TPT)", "F", f_carrier, p_carrier)

    worst_plant = df.groupby("Plant Code")["Delivery_Delay"].mean().idxmax()
    is_worst = df["Plant Code"] == worst_plant
    t_stat, p_plant = stats.ttest_ind(
        df.loc[is_worst, "Delivery_Delay"], df.loc[~is_worst, "Delivery_Delay"],
        equal_var=False, alternative="greater",
    )
    show_result(f"TEST 2: WELCH T-TEST ({worst_plant} delay vs rest of network)", "t", t_stat,
                p_plant)

    pair = df[["Weight", TARGET]].dropna()
    r_weight, p_weight = stats.pearsonr(pair["Weight"], pair[TARGET])
    show_result("TEST 3: PEARSON CORRELATION (Weight vs Total Lead Time)", "r", r_weight,
                p_weight)
    strength = "weak" if abs(r_weight) < 0.3 else "moderate" if abs(r_weight) < 0.7 else "strong"
    print(f"Strength of the linear relationship: {strength}")

    port_groups = [g[TARGET].dropna() for _, g in df.groupby("Origin Port")]
    f_port, p_port = stats.f_oneway(*port_groups)
    show_result("TEST 4: ONE-WAY ANOVA (Origin Port vs Total Lead Time)", "F", f_port, p_port)
    print("=" * 70)

    return {"carrier": (f_carrier, p_carrier), "plant": (t_stat, p_plant),
            "weight": (r_weight, p_weight), "port": (f_port, p_port), "worst_plant": worst_plant}


tests = run_hypothesis_tests(cleaned)
worst_plant = tests["worst_plant"]

summary = pd.DataFrame(
    [
        ["ANOVA: Carrier vs TPT", "F", *tests["carrier"]],
        [f"Welch t-test: {worst_plant} delay vs rest", "t", *tests["plant"]],
        ["Pearson: Weight vs Total Lead Time", "r", *tests["weight"]],
        ["ANOVA: Origin Port vs Total Lead Time", "F", *tests["port"]],
    ],
    columns=["Test", "Statistic type", "Statistic", "P-value"],
)
summary["Decision"] = summary["P-value"].map(verdict)
summary["Statistic"] = summary["Statistic"].round(4)
summary["P-value"] = summary["P-value"].map("{:.3e}".format)
display(summary)

In [ ]:
def plot_hypothesis_evidence(df: pd.DataFrame, plant: str) -> None:
    """Visualise the carrier ANOVA, the plant t-test and the weight correlation."""
    fig = plt.figure(figsize=(18, 6), layout="constrained")
    grid = fig.add_gridspec(1, 3)

    ax = fig.add_subplot(grid[0, 0])
    order = list(df.groupby("Carrier")["TPT"].median().sort_values().index)
    box = ax.boxplot([df.loc[df["Carrier"] == c, "TPT"] for c in order], patch_artist=True)
    for patch in box["boxes"]:
        patch.set_facecolor(COLORS["muted"])
    ax.set_xticklabels(order)
    ax.set_title("ANOVA: TPT by carrier", fontsize=13, fontweight="bold")
    ax.set_ylabel("Transit processing time (days)")

    ax = fig.add_subplot(grid[0, 1])
    is_plant = df["Plant Code"] == plant
    groups = {plant: df.loc[is_plant, "Delivery_Delay"],
              "Rest of network": df.loc[~is_plant, "Delivery_Delay"]}
    ax.bar(list(groups), [g.mean() for g in groups.values()],
           yerr=[1.96 * stats.sem(g) for g in groups.values()], capsize=8,
           color=[COLORS["danger"], COLORS["muted"]])
    ax.set_title(f"Welch t-test: {plant} vs rest of network", fontsize=13, fontweight="bold")
    ax.set_ylabel("Average delivery delay (days), 95% CI")

    ax = fig.add_subplot(grid[0, 2])
    sns.regplot(data=df, x="Weight", y=TARGET, scatter_kws={"alpha": 0.3, "s": 15,
                "color": COLORS["primary"]}, line_kws={"color": COLORS["accent"], "linewidth": 3},
                ax=ax)
    ax.set_title("Pearson: weight vs lead time", fontsize=13, fontweight="bold")
    plt.suptitle("Statistical hypothesis validation", fontsize=18, fontweight="bold")
    plt.show()


plot_hypothesis_evidence(cleaned, worst_plant)

In [ ]:
# Tukey HSD: which specific plants differ in mean TPT?
tukey = pairwise_tukeyhsd(endog=cleaned["TPT"], groups=cleaned["Plant Code"], alpha=ALPHA)
print(tukey.summary())

fig = tukey.plot_simultaneous(figsize=(12, 7))
ax = fig.gca()
ax.set_title("Post-hoc: plant differences in mean TPT (Tukey HSD)", fontsize=15, fontweight="bold")
ax.set_xlabel("Mean transit processing time (days)")
ax.set_ylabel("Plant code")
plt.show()

## 6. Predictive modelling
Target: `Total_Lead_Time`. Features: weight, carrier, origin port, plant, service level and unit quantity (categoricals one-hot encoded). One 80/20 split is shared by every model, so RMSE and R² rank the models identically. The leaderboard is **computed here**, not typed in.

*Linear Regression and OLS fit the same linear model on the same features, so their test metrics are identical. OLS is included for its inference table (coefficients and p-values).*

In [ ]:
model_data = cleaned[FEATURES + [TARGET]].dropna()
X = pd.get_dummies(model_data[FEATURES], drop_first=True).astype(float)
y = model_data[TARGET]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=SEED
)
print(f"Features after encoding: {X.shape[1]} | train: {len(X_train):,} | test: {len(X_test):,}")

results = []

# Model 1: Linear Regression (baseline)
lr_model = LinearRegression().fit(X_train, y_train)
lr_pred = lr_model.predict(X_test)
results.append(evaluate_model("Linear Regression", y_test, lr_pred))

# Model 2: Random Forest (captures non-linear patterns)
rf_model = RandomForestRegressor(
    n_estimators=200, max_depth=10, min_samples_split=5, random_state=SEED, n_jobs=-1
).fit(X_train, y_train)
rf_pred = rf_model.predict(X_test)
results.append(evaluate_model("Random Forest", y_test, rf_pred))

# Model 3: OLS with statsmodels (inference table), evaluated on the same test set
ols_model = sm.OLS(y_train, sm.add_constant(X_train, has_constant="add")).fit()
ols_pred = ols_model.predict(sm.add_constant(X_test, has_constant="add"))
results.append(evaluate_model("OLS Regression", y_test, ols_pred))
print(ols_model.summary())

In [ ]:
# Model 4: XGBoost baseline
xgb_model = xgboost.XGBRegressor(
    n_estimators=200, learning_rate=0.1, max_depth=6, random_state=SEED, n_jobs=-1
).fit(X_train, y_train)
results.append(evaluate_model("XGBoost", y_test, xgb_model.predict(X_test)))

# Model 5: XGBoost tuned with 5-fold GridSearchCV
PARAM_GRID = {
    "n_estimators": [100, 200, 300],
    "max_depth": [4, 6, 8],
    "learning_rate": [0.01, 0.05, 0.1],
    "subsample": [0.8, 1.0],
    "colsample_bytree": [0.8, 1.0],
}
if FAST_MODE:
    PARAM_GRID = {key: values[:1] for key, values in PARAM_GRID.items()}

grid_search = GridSearchCV(
    estimator=xgboost.XGBRegressor(random_state=SEED, n_jobs=-1),
    param_grid=PARAM_GRID, cv=5, scoring="neg_mean_absolute_error", n_jobs=-1, verbose=1,
)
grid_search.fit(X_train, y_train)
print("Best parameters:", grid_search.best_params_)
results.append(evaluate_model("XGBoost (Optimized)", y_test,
                              grid_search.best_estimator_.predict(X_test)))

In [ ]:
leaderboard = pd.DataFrame(results).sort_values("RMSE (Days)").reset_index(drop=True)
champion = leaderboard.loc[0, "Model"]
leaderboard["Status"] = ["Champion (lowest RMSE)"] + ["Alternative"] * (len(leaderboard) - 1)
print("=" * 75)
print(f"{'MODEL LEADERBOARD (computed on the shared test set)':^75}")
print("=" * 75)
display(leaderboard.round(4))
champion_r2 = leaderboard.loc[0, "R2"]
print(f"Champion: {champion}. It explains {champion_r2:.1%} of lead-time variance, "
      "so most variation comes from factors outside this dataset.")


def plot_model_results() -> None:
    """Plot leaderboard metrics, Random Forest fit and RF feature importance."""
    fig, axes = plt.subplots(1, 2, figsize=(14, 5), layout="constrained")
    colors = [COLORS["primary"] if m == champion else COLORS["grey"] for m in leaderboard["Model"]]
    for ax, column, title in zip(axes, ["RMSE (Days)", "R2"],
                                 ["RMSE (lower is better)", "R2 (higher is better)"]):
        ax.bar(leaderboard["Model"], leaderboard[column], color=colors)
        label_bars(ax, "%.3f")
        ax.set_title(title, fontweight="bold")
        ax.tick_params(axis="x", rotation=30)
    axes[1].set_ylim(0, 1)
    plt.suptitle("Model leaderboard", fontsize=16, fontweight="bold")
    plt.show()

    fig = plt.figure(figsize=(18, 6), layout="constrained")
    grid = fig.add_gridspec(1, 3)
    for i, (name, pred, color) in enumerate([("Linear Regression", lr_pred, COLORS["muted"]),
                                             ("Random Forest", rf_pred, COLORS["primary"])]):
        ax = fig.add_subplot(grid[0, i])
        ax.scatter(y_test, pred, alpha=0.4, color=color)
        ax.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], "r--", lw=2)
        ax.set_title(f"{name}: actual vs predicted", fontsize=13, fontweight="bold")
        ax.set_xlabel("Actual lead time (days)")
        ax.set_ylabel("Predicted lead time (days)")
    ax = fig.add_subplot(grid[0, 2])
    top = pd.Series(rf_model.feature_importances_, index=X.columns).nlargest(10).sort_values()
    ax.barh(top.index, top.values, color=COLORS["accent"])
    ax.set_title("Random Forest: top 10 feature drivers", fontsize=13, fontweight="bold")
    ax.set_xlabel("Relative importance")
    plt.show()

    fig, axes = plt.subplots(1, 2, figsize=(16, 6), layout="constrained")
    axes[0].scatter(ols_model.fittedvalues, ols_model.resid, alpha=0.5, color=COLORS["primary"])
    axes[0].axhline(0, color=COLORS["danger"], linestyle="--", lw=2)
    axes[0].set_title("OLS: residuals vs fitted values", fontweight="bold")
    axes[0].set_xlabel("Predicted lead time (days)")
    axes[0].set_ylabel("Residual")
    sns.histplot(ols_model.resid, kde=True, color=COLORS["muted"], ax=axes[1])
    axes[1].axvline(0, color=COLORS["danger"], linestyle="--", lw=2)
    axes[1].set_title("OLS: distribution of residuals", fontweight="bold")
    plt.show()


plot_model_results()

## 7. Findings and recommendations

In [ ]:
def build_evidence_table(df: pd.DataFrame) -> pd.DataFrame:
    """Combine computed results into one evidence table. Every figure comes from this run."""
    port_share = df["Origin Port"].value_counts(normalize=True)
    plant_tpt = df.groupby("Plant Code")["TPT"].mean().sort_values(ascending=False)
    slowest = plant_tpt[np.isclose(plant_tpt, plant_tpt.max())].index.tolist()
    carrier_tpt = df.groupby("Carrier")["TPT"].mean().drop("V44_3", errors="ignore")
    carrier_share = df["Carrier"].value_counts(normalize=True)
    fastest = carrier_tpt.idxmin()

    rows = [
        ("Port concentration",
         f"{port_share.index[0]} handles {port_share.iloc[0]:.1%} of shipments "
         f"(port ANOVA p = {tests['port'][1]:.2e}).",
         "Pilot routing some non-urgent shipments through a secondary port and track lead time. "
         "The aim is resilience, not speed."),
        ("Carrier choice",
         f"Carrier affects TPT (ANOVA p = {tests['carrier'][1]:.2e}). {fastest} has the lowest "
         f"mean TPT among commercial carriers ({carrier_tpt.min():.2f} days) but handles "
         f"{carrier_share[fastest]:.1%} of shipments.",
         "Pilot more volume with the fastest carrier before consolidating. "
         "Exclude the customer-arranged carrier (V44_3)."),
        ("Plant workload",
         f"Highest mean TPT: {', '.join(slowest)} ({plant_tpt.max():.2f} days). Lowest: "
         f"{plant_tpt.idxmin()} ({plant_tpt.min():.2f} days).",
         "Review workload balance and lead-time-based safety stock. "
         "Validate with capacity and cost data."),
        ("Shipment weight",
         f"Weak linear link to lead time (r = {tests['weight'][0]:.3f}).",
         "Do not build routing rules on weight alone."),
        ("Prediction",
         f"{champion}: RMSE {leaderboard.loc[0, 'RMSE (Days)']:.2f} days, R2 {champion_r2:.2f}.",
         "Use as a baseline. Add congestion and dated data before operational use."),
    ]
    return pd.DataFrame(rows, columns=["Area", "Evidence from this run", "Suggested action"])


evidence = build_evidence_table(cleaned)
pd.set_option("display.max_colwidth", None)
display(evidence)

## 8. Limitations and next steps
**Limitations**
- Single order date: no trend, seasonality or time-based validation.
- Small groups (PORT05 has one record; some plants have few records), so those results are indicative only.
- R² is modest, so external factors such as congestion and weather are not captured.
- Only the `OrderList` table was used. Freight cost, warehouse capacity and warehouse cost tables were not analysed, so **no cost saving is claimed**.
- Port, plant and carrier labels are anonymised codes; any mapping to real locations is an assumption.

**Next steps**
- Use the freight-rate, capacity and cost tables to size the impact of each recommendation.
- Add dated data and validate with a time-based split plus cross-validation.
- Build a carrier cost-and-performance scorecard.

*Dataset credit: Supply Chain Logistics Problem on Kaggle (owner `anisseezzebdi`). Check the dataset page for its license.*